# 多分类神经网络在量子关联分类中的应用研究

Research on the application of multi-classification neural network in quantum correlation classification (MCNN-QCC)

## 概要

本课题将通过知名深度学习框架Pytorch搭建神经网络，分析QCC数据集，观察训练误差和验证误差随训练代数提高的变化。

徐浩轩 集美大学 2023-12

GitHub：[ollyhsu/MCNN-QCC](https://github.com/ollyhsu/MCNN-QCC)


# 环境准备

## 下载Pytorch

打开浏览器，访问 [PyTorch官方网站](https://pytorch.org/)

在官网首页，你会看到一个“Get Started”（开始使用）的按钮或类似的链接，点击它

接下来，你可能需要选择一些选项，比如选择你的操作系统、安装方式（例如pip或conda）、Python版本等。根据你的需求进行选择，并点击相关选项

完成上述步骤后，网站会为你生成一个相应的安装命令。复制这个命令，它会类似于：

```bash
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

## 验证环境

你可以运行以下代码单元来验证PyTorch的安装是否成功，并检查CUDA是否可用以及CUDA的版本：

In [ ]:
import os
import tools as tl
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
# from tqdm import tqdm

%matplotlib inline
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
label_dic = {0: 'fully-separable states', 1: 'two-separable states', 2: 'entangled states'}

In [ ]:
# 检查CUDA是否可用
cuda_available = torch.cuda.is_available()
print("CUDA Available: ", cuda_available)

if cuda_available:
    # 打印CUDA设备数量
    device_count = torch.cuda.device_count()
    print("CUDA Device Count: ", device_count)

    # 打印CUDA版本
    cuda_version = torch.version.cuda
    print("CUDA Version: ", cuda_version)

运行此代码单元后，如果你看到输出显示CUDA可用、设备数量以及CUDA版本，则说明PyTorch已经成功安装，并且CUDA也可用

# 数据集准备

## 生成数据集

运行`genDataset.py` 文件，生成数据集至 `dataset` 目录下

数据集标签说明：

`0` for fully-separable states, `1` for two-separable states, `2` for entangled states

## 导入数据集

In [ ]:
# 准备数据
train_mixes = np.load('dataset/qcc_train_set_mixes.npy')
train_labels = np.load('dataset/qcc_train_set_labels.npy')
test_mixes = np.load('dataset/qcc_test_set_mixes.npy')
test_labels = np.load('dataset/qcc_test_set_labels.npy')

# Convert to PyTorch tensors
train_mixes_tensor = torch.tensor(train_mixes, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)

test_mixes_tensor = torch.tensor(test_mixes, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)

# Extract real and imaginary parts
train_mixes_real = train_mixes_tensor[:, :, :, 0]
train_mixes_imag = train_mixes_tensor[:, :, :, 1]

test_mixes_real = test_mixes_tensor[:, :, :, 0]
test_mixes_imag = test_mixes_tensor[:, :, :, 1]

# Complex tensor
train_mixes_complex = torch.stack((train_mixes_real, train_mixes_imag), dim=1)
test_mixes_complex = torch.stack((test_mixes_real, test_mixes_imag), dim=1)

# Create PyTorch datasets
train_dataset = TensorDataset(train_mixes_complex, train_labels_tensor)
test_dataset = TensorDataset(test_mixes_complex, test_labels_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=100, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=100, shuffle=True)

# print("Train Mixes Tensor Shape: ", train_mixes_tensor.shape)
# print("Train Labels Tensor Shape: ", train_labels_tensor.shape)
# print("Test Mixes Tensor Shape: ", test_mixes_tensor.shape)
# print("Test Labels Tensor Shape: ", test_labels_tensor.shape)

print("Train Mixes ComplexShape: ", train_mixes_complex.shape)
print("Test Mixes ComplexShape: ", test_mixes_complex.shape)

print("Train Dataset Size: ", len(train_dataset))
print("Test Dataset Size: ", len(test_dataset))
print("Train Dataloader Size: ", len(train_dataloader))
print("Test Dataloader Size: ", len(test_dataloader))

# 多层感知机模型

## 多层感知机

神经网络的输入为 2 * 8 * 8 = 128 的张量

第一个隐含层包含256个神经元

第二个隐含层包含多128个神经元

第三个隐含层包含64个神经元

输出层输出3个结果，对应数据集量子态的3种分类

In [ ]:
# 定义四层多层感知机模型
class MLPModel(nn.Module):
    def __init__(self):
        super(MLPModel, self).__init__()
        self.fc1 = nn.Linear(128, 256)  # 第一个隐藏层，输入大小为128，输出大小为256
        self.fc2 = nn.Linear(256, 128)  # 第二个隐藏层，输入大小为256，输出大小为128
        self.fc3 = nn.Linear(128, 64)  # 第三个隐藏层，输入大小为128，输出大小为64
        self.fc4 = nn.Linear(64, 3)  # 输出层，输入大小为64，输出大小为3（类别数量）

    def forward(self, x):
        x = torch.flatten(x, 1)  # 将输入展平成一维张量
        x = F.relu(self.fc1(x))  # 第一个隐藏层的激活函数为ReLU
        x = F.relu(self.fc2(x))  # 第二个隐藏层的激活函数为ReLU
        x = F.relu(self.fc3(x))  # 第三个隐藏层的激活函数为ReLU
        x = self.fc4(x)  # 输出层不需要激活函数，因为后面会使用交叉熵损失函数
        return x

In [ ]:
# 实例化模型
model_mlp = MLPModel()

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)

# 训练模型
num_epochs = 20

mlp_train_losses, mlp_test_losses, mlp_train_accs, mlp_test_accs = tl.model_trainer(num_epochs, train_dataloader, test_dataloader, model_mlp, criterion, optimizer)

In [ ]:
# 保存模型
# torch.save(model_mlp.state_dict(), 'models/model_mlp.pth', _use_new_zipfile_serialization=False)

验证模型效果

绘制训练误差和测试误差随学习次数增加的变化

In [ ]:
# Training and Test Loss
tl.draw_loss(mlp_train_losses, mlp_test_losses)

In [ ]:
# Accuracy Curve
tl.draw_acc(mlp_train_accs, mlp_test_accs)

In [ ]:
# # 加载模型
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model_mlp = MLPModel()
# model_mlp.load_state_dict(torch.load('models/model_mlp.pth', map_location=device))
# model_mlp.to(device)

In [ ]:
conf_matrix_percent_mlp, class_accuracy_mlp, precision_mlp, recall_mlp, f1_mlp = tl.model_tester(test_dataloader, model_mlp)

In [ ]:
# Confusion Matrix
tl.draw_confusion(conf_matrix_percent_mlp, label_dic)

In [ ]:
# Classification accuracy
tl.draw_class_acc(class_accuracy_mlp, label_dic)

In [ ]:
# 数据
precision = precision_mlp
recall = recall_mlp
f1 = f1_mlp
class_accuracy = class_accuracy_mlp

# 输出表头
print("Type\tPrecision/%\tRecall/%\tAccuracy/%\tF1")

# 输出每个类别的指标
for i, label in enumerate(label_dic.values()):
    print(f"{label}\t{precision[i] * 100:.2f}\t{recall[i] * 100:.2f}\t{class_accuracy[i]:.2f}\t{f1[i]:.2f}")

In [ ]:
# Precision, Recall and F1 Score for each Class
tl.draw_prf1(precision_mlp, recall_mlp, f1_mlp, label_dic)

## 采用Dropout防止过拟合

In [ ]:
class MLPModelWithDropout(nn.Module):
    def __init__(self):
        super(MLPModelWithDropout, self).__init__()
        self.fc1 = nn.Linear(8 * 8 * 2, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 3)
        self.dropout = nn.Dropout(p=0.5)  # 添加 Dropout 层，p 参数表示丢弃概率

    def forward(self, x):
        x = x.view(-1, 8 * 8 * 2)  # 展平输入数据
        x = F.relu(self.fc1(x))
        x = self.dropout(x)  # 在第一个全连接层后添加 Dropout
        x = F.relu(self.fc2(x))
        x = self.dropout(x)  # 在第二个全连接层后添加 Dropout
        x = F.relu(self.fc3(x))
        x = self.dropout(x)  # 在第三个全连接层后添加 Dropout
        x = self.fc4(x)
        return x

In [ ]:
# 实例化模型
model_mlpd = MLPModelWithDropout()

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mlpd.parameters(), lr=0.001)

# 训练模型
num_epochs = 20

mlpd_train_losses, mlpd_test_losses, mlpd_train_accs, mlpd_test_accs = tl.model_trainer(num_epochs, train_dataloader, test_dataloader, model_mlpd, criterion, optimizer)

In [ ]:
# 保存模型
# torch.save(model_mlpd.state_dict(), 'models/model_mlpd.pth', _use_new_zipfile_serialization=False)

验证模型效果

In [ ]:
# Training and Test Loss
tl.draw_loss(mlpd_train_losses, mlpd_test_losses)

In [ ]:
# Accuracy Curve
tl.draw_acc(mlpd_train_accs, mlpd_test_accs)

In [ ]:
# # 加载模型
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model_mlpd = MLPModelWithDropout()
# model_mlpd.load_state_dict(torch.load('models/model_mlpd.pth', map_location=device))
# model_mlpd.to(device)

In [ ]:
conf_matrix_percent_mlpd, class_accuracy_mlpd, precision_mlpd, recall_mlpd, f1_mlpd = tl.model_tester(test_dataloader, model_mlpd)

In [ ]:
# Confusion Matrix
tl.draw_confusion(conf_matrix_percent_mlpd, label_dic)

In [ ]:
# Classification accuracy
tl.draw_class_acc(class_accuracy_mlpd, label_dic)

In [ ]:
# 数据
precision = precision_mlpd
recall = recall_mlpd
f1 = f1_mlpd
class_accuracy = class_accuracy_mlpd

# 输出表头
print("Type\tPrecision/%\tRecall/%\tAccuracy/%\tF1")

# 输出每个类别的指标
for i, label in enumerate(label_dic.values()):
    print(f"{label}\t{precision[i] * 100:.2f}\t{recall[i] * 100:.2f}\t{class_accuracy[i]:.2f}\t{f1[i]:.2f}")

In [ ]:
tl.draw_prf1(precision_mlpd, recall_mlpd, f1_mlpd, label_dic)

对比图

In [ ]:
from matplotlib import pyplot as plt
# 生成示例数据
epochs = range(1, 11)

# 绘制图像
plt.figure(figsize=(10, 6))

# MLP的Train loss 和 test loss
plt.plot(epochs, mlp_train_losses, 'bo-', label='MLP Train Loss')
plt.plot(epochs, mlp_test_losses, 'yo--', label='MLP Test Loss')

# MLP With Dropout的Train loss 和 test loss
plt.plot(epochs, mlpd_train_losses, 'rs-', label='MLP Dropout Train Loss')
plt.plot(epochs, mlpd_test_losses, 'ms--', label='MLP Dropout Test Loss')

plt.title('MLP Model Train and Test Loss by Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from matplotlib import pyplot as plt

# 生成示例数据
epochs = range(1, 11)

# 绘制图像
plt.figure(figsize=(10, 6))

# mlp_train_accs 和 mlp_test_accs
plt.plot(epochs, mlp_train_accs, 'bo-', label='MLP Train Acc')
plt.plot(epochs, mlp_test_accs, 'yo--', label='MLP Test Acc')

# mlpd_train_accs 和 mlpd_test_accs
plt.plot(epochs, mlpd_train_accs, 'rs-', label='MLP Dropout Train Acc')
plt.plot(epochs, mlpd_test_accs, 'ms--', label='MLP Dropout Test Acc')

plt.title('MLP Model Train and Test Accuracies by Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 示例混淆矩阵数据
conf_matrix1 = np.array(conf_matrix_percent_mlp)
conf_matrix2 = np.array(conf_matrix_percent_mlpd)

# 标签
labels = label_dic.values()

# 创建图表
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

# 绘制第一个混淆矩阵
im1 = axs[0].imshow(conf_matrix1, cmap='Blues')

# 绘制第二个混淆矩阵
im2 = axs[1].imshow(conf_matrix2, cmap='Blues')

# 添加颜色条
fig.colorbar(im1, ax=axs[0])
fig.colorbar(im2, ax=axs[1])

# 在每个单元格中标注数据
for i in range(conf_matrix1.shape[0]):
    for j in range(conf_matrix1.shape[1]):
        axs[0].text(j, i, '{:.2f}'.format(conf_matrix1[i, j]), ha='center', va='center', color='red')

for i in range(conf_matrix2.shape[0]):
    for j in range(conf_matrix2.shape[1]):
        axs[1].text(j, i, '{:.2f}'.format(conf_matrix2[i, j]), ha='center', va='center', color='red')

# 设置图表标题和标签
axs[0].set_title('Confusion Matrix MLP')
axs[0].set_xticks(np.arange(len(labels)))
axs[0].set_xticklabels(labels, rotation=45)
axs[0].set_yticks(np.arange(len(labels)))
axs[0].set_yticklabels(labels)

axs[1].set_title('Confusion Matrix MLP with Dropout ')
axs[1].set_xticks(np.arange(len(labels)))
axs[1].set_xticklabels(labels, rotation=45)
axs[1].set_yticks(np.arange(len(labels)))
axs[1].set_yticklabels(labels)

# 显示图表
plt.tight_layout()
plt.show()


In [ ]:
# 分类预测
x = np.arange(len(label_dic))
width = 0.35
label_dic = {0: 'fully-separable states', 1: 'two-separable states', 2: 'entangled states'}
fig, ax = plt.subplots(figsize=(10, 6))
bars_1 = ax.bar(x - width/2, class_accuracy_mlp, width, label='MLP')
bars_2 = ax.bar(x + width/2, class_accuracy_mlpd, width, label='MLPD')

# 添加标签、标题
ax.set_xlabel('Class Labels')
ax.set_ylabel('Accuracy (%)')
ax.set_title('MLP Model Class-wise Accuracy Comparison')
ax.set_xticks(x)
ax.set_xticklabels(label_dic.values(), rotation=15)


# 添加数值标签
def autolabel(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate('%.2f' % height,
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 2),
                    textcoords="offset points",
                    ha='center', va='bottom')


autolabel(bars_1)
autolabel(bars_2)
# 移动图例到右边
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

# 卷积神经网络模型

## LeNet-5神经网络

对于该数据集，可以设计一个简单的LeNet-5结构，适用于处理8x8的复数矩阵。以下是LeNet-5结构示例：

1. **卷积层1(Convolutional Layer 1)**：
   - 输入大小：2x8x8（8x8的复数矩阵，2表示实部和虚部）
   - 卷积核大小：5x5
   - 卷积核数量：6
   - 激活函数：ReLU
   - 输出大小：4x4x6

2. **池化层1(Pooling Layer 1)**：
   - 池化类型：最大池化（Max Pooling）
   - 池化窗口大小：2x2
   - 输出大小：6x2x2

3. **卷积层2(Convolutional Layer 2)**：
   - 输入大小：6x2x2
   - 卷积核大小：2x2
   - 卷积核数量：16
   - 激活函数：ReLU
   - 输出大小：16x1x1

4. **池化层2(Pooling Layer 2)**：
   - 池化类型：最大池化（Max Pooling）
   - 池化窗口大小：1x1
   - 输出大小：16x1x1

5. **全连接层1(Fully Connected Layer 1)**：
   - 输入大小：16x1x1
   - 输出大小：120
   - 激活函数：ReLU

6. **全连接层2(Fully Connected Layer 2)**：
   - 输入大小：120
   - 输出大小：84
   - 激活函数：ReLU

7. **输出层(Output Layer)**：
   - 输入大小：84
   - 输出大小：3（对应三个类别）


### 模型设计

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # Define the layers
        self.conv1 = nn.Conv2d(in_channels=2, out_channels=6, kernel_size=5, stride=1, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 最大池化层
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=2, stride=1, padding=0)
        self.pool2 = nn.MaxPool2d(kernel_size=1, stride=1)  # 最大池化层
        self.fc1 = nn.Linear(16 * 1 * 1, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3)  # Three classes

    def forward(self, x):
        # Apply convolutional layers with ReLU activation and max pooling
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        # Flatten the output for fully connected layers
        # x = torch.flatten(x, 1)  # 将输入展平成一维张量
        x = x.view(-1, 16 * 1 * 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
# 实例化模型
model_le = LeNet5()

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_le.parameters(), lr=0.001)

# 训练模型
num_epochs = 20
le_train_losses, le_test_losses, le_train_accs, le_test_accs = tl.model_trainer(num_epochs, train_dataloader, test_dataloader, model_le, criterion, optimizer)

In [ ]:
# # 保存模型
# torch.save(model_le.state_dict(), 'models/model_le.pth', _use_new_zipfile_serialization=False)

验证模型效果

In [ ]:
# Training and Test Loss
tl.draw_loss(le_train_losses, le_test_losses)

In [ ]:
# Accuracy Curve
tl.draw_acc(le_train_accs, le_test_accs)

In [ ]:
# # 加载模型
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model_le = LeNet5()
# model_le.load_state_dict(torch.load('models/model_le.pth', map_location=device))
# model_le.to(device)

In [ ]:
conf_matrix_percent_le, class_accuracy_le, precision_le, recall_le, f1_le = tl.model_tester(test_dataloader, model_le)

In [ ]:
# Confusion Matrix
tl.draw_confusion(conf_matrix_percent_le, label_dic)

In [ ]:
# Classification accuracy
tl.draw_class_acc(class_accuracy_le, label_dic)

In [ ]:
# 数据
precision = precision_le
recall = recall_le
f1 = f1_le
class_accuracy = class_accuracy_le

# 输出表头
print("Type\tPrecision/%\tRecall/%\tAccuracy/%\tF1")

# 输出每个类别的指标
for i, label in enumerate(label_dic.values()):
    print(f"{label}\t{precision[i] * 100:.2f}\t{recall[i] * 100:.2f}\t{class_accuracy[i]:.2f}\t{f1[i]:.2f}")

In [ ]:
# Precision, Recall and F1 Score for each Class
tl.draw_prf1(precision_le, recall_le, f1_le, label_dic)

### 采用Dropout防过拟合

In [ ]:
class LeNet5WithDropout(nn.Module):
    def __init__(self):
        super(LeNet5WithDropout, self).__init__()
        # Define the layers
        self.conv1 = nn.Conv2d(in_channels=2, out_channels=6, kernel_size=5, stride=1, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 最大池化层
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=2, stride=1, padding=0)
        self.pool2 = nn.MaxPool2d(kernel_size=1, stride=1)  # 最大池化层
        self.fc1 = nn.Linear(16 * 1 * 1, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3)  # Three classes
        self.dropout = nn.Dropout(p=0.5)  # 添加 Dropout 层，p 参数表示丢弃概率

    def forward(self, x):
        # Apply convolutional layers with ReLU activation and max pooling
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        # Flatten the output for fully connected layers
        # x = torch.flatten(x, 1)  # 将输入展平成一维张量
        x = x.view(-1, 16 * 1 * 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [ ]:
# 实例化模型
model_led = LeNet5WithDropout()

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_led.parameters(), lr=0.001)

# 训练模型
num_epochs = 20
led_train_losses, led_test_losses, led_train_accs, led_test_accs = tl.model_trainer(num_epochs, train_dataloader, test_dataloader, model_led, criterion, optimizer)

In [ ]:
# # 保存模型
# torch.save(model_led.state_dict(), 'models/model_led.pth', _use_new_zipfile_serialization=False)

In [ ]:
# Training and Test Loss
tl.draw_loss(led_train_losses, led_test_losses)

In [ ]:
# Accuracy Curve
tl.draw_acc(led_train_accs, led_test_accs)

In [ ]:
# # 加载模型
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model_led = LeNet5WithDropout()
# model_led.load_state_dict(torch.load('models/model_led.pth', map_location=device))
# model_led.to(device)

In [ ]:
conf_matrix_percent_led, class_accuracy_led, precision_led, recall_led, f1_led = tl.model_tester(test_dataloader, model_led)

In [ ]:
# Confusion Matrix
tl.draw_confusion(conf_matrix_percent_led, label_dic)

In [ ]:
# Classification accuracy
tl.draw_class_acc(class_accuracy_led, label_dic)

In [ ]:
# 数据
precision = precision_led
recall = recall_led
f1 = f1_led
class_accuracy = class_accuracy_led

# 输出表头
print("Type\tPrecision/%\tRecall/%\tAccuracy/%\tF1")

# 输出每个类别的指标
for i, label in enumerate(label_dic.values()):
    print(f"{label}\t{precision[i] * 100:.2f}\t{recall[i] * 100:.2f}\t{class_accuracy[i]:.2f}\t{f1[i]:.2f}")

In [ ]:
# Precision, Recall and F1 Score for each Class
tl.draw_prf1(precision_led, recall_led, f1_led, label_dic)

对比图

In [ ]:
from matplotlib import pyplot as plt

# 生成示例数据
epochs = range(1, 11)

# 绘制图像
plt.figure(figsize=(10, 6))

# LeNet5的Train loss 和 test loss
plt.plot(epochs, le_train_losses, 'bo-', label='LeNet5 Train Loss')
plt.plot(epochs, le_test_losses, 'yo--', label='LeNet5 Test Loss')

# LeNet5 With Dropout的Train loss 和 test loss
plt.plot(epochs, led_train_losses, 'rs-', label='LeNet5 Dropout Train Loss')
plt.plot(epochs, led_test_losses, 'ms--', label='LeNet5 Dropout Test Loss')

plt.title('LeNet5 Model Train and Test Loss by Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from matplotlib import pyplot as plt

# 生成示例数据
epochs = range(1, 11)

# 绘制图像
plt.figure(figsize=(10, 6))

# le_train_accs 和 le_test_accs
plt.plot(epochs, le_train_accs, 'bo-', label='LeNet5 Train Acc')
plt.plot(epochs, le_test_accs, 'yo--', label='LeNet5 Test Acc')

# led_train_accs 和 led_test_accs
plt.plot(epochs, led_train_accs, 'rs-', label='LeNet5 Dropout Train Acc')
plt.plot(epochs, led_test_accs, 'ms--', label='LeNet5 Dropout Test Acc')

plt.title('LeNet5 Model Train and Test Accuracies by Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 示例混淆矩阵数据
conf_matrix1 = np.array(conf_matrix_percent_le)
conf_matrix2 = np.array(conf_matrix_percent_led)

# 标签
labels = label_dic.values()

# 创建图表
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

# 绘制第一个混淆矩阵
im1 = axs[0].imshow(conf_matrix1, cmap='Blues')

# 绘制第二个混淆矩阵
im2 = axs[1].imshow(conf_matrix2, cmap='Blues')

# 添加颜色条
fig.colorbar(im1, ax=axs[0])
fig.colorbar(im2, ax=axs[1])

# 在每个单元格中标注数据
for i in range(conf_matrix1.shape[0]):
    for j in range(conf_matrix1.shape[1]):
        axs[0].text(j, i, '{:.2f}'.format(conf_matrix1[i, j]), ha='center', va='center', color='red')

for i in range(conf_matrix2.shape[0]):
    for j in range(conf_matrix2.shape[1]):
        axs[1].text(j, i, '{:.2f}'.format(conf_matrix2[i, j]), ha='center', va='center', color='red')

# 设置图表标题和标签
axs[0].set_title('Matrix LeNet5')
axs[0].set_xticks(np.arange(len(labels)))
axs[0].set_xticklabels(labels, rotation=45)
axs[0].set_yticks(np.arange(len(labels)))
axs[0].set_yticklabels(labels)

axs[1].set_title('Matrix LeNet5 with Dropout ')
axs[1].set_xticks(np.arange(len(labels)))
axs[1].set_xticklabels(labels, rotation=45)
axs[1].set_yticks(np.arange(len(labels)))
axs[1].set_yticklabels(labels)

# 显示图表
plt.tight_layout()
plt.show()

In [ ]:
# 分类预测
x = np.arange(len(label_dic))
width = 0.35
label_dic = {0: 'fully-separable states', 1: 'two-separable states', 2: 'entangled states'}
fig, ax = plt.subplots(figsize=(10, 6))
bars_1 = ax.bar(x - width / 2, class_accuracy_le, width, label='LeNet5')
bars_2 = ax.bar(x + width / 2, class_accuracy_led, width, label='LeNet5 Dropout')

# 添加标签、标题
ax.set_xlabel('Class Labels')
ax.set_ylabel('Accuracy (%)')
ax.set_title('LeNet5 Model Class-wise Accuracy Comparison')
ax.set_xticks(x)
ax.set_xticklabels(label_dic.values(), rotation=15)


# 添加数值标签
def autolabel(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate('%.2f' % height,
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 2),
                    textcoords="offset points",
                    ha='center', va='bottom')


autolabel(bars_1)
autolabel(bars_2)
# 移动图例到右边
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()